<a href="https://colab.research.google.com/github/ghazal-mohammad/ir-system/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IR System — Preprocessing
**Step 2:** Clean and normalize text for both datasets.

In [1]:
# Mount Google Drive to save processed data
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Save dir:', SAVE_DIR)

Mounted at /content/drive
Save dir: /content/drive/MyDrive/ir_system_data


In [2]:
!pip install ir-datasets nltk -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('ready')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.5 MB/s eta 0:00:00
ready


In [3]:
# Clone the repo to get the services
import os
if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull

import sys
sys.path.insert(0, '/content/ir-system')
print('repo ready')

Cloning into '/content/ir-system'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 31 (delta 10), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 20.76 KiB | 10.38 MiB/s, done.
Resolving deltas: 100% (10/10), done.
repo ready


In [4]:
from services.preprocessing_service import preprocess, preprocess_to_string

# Quick test
test = 'Patient is a 45-year-old man with history of diabetes and hypertension!!!'
print('Original  :', test)
print('Processed :', preprocess(test))
print('As string :', preprocess_to_string(test))

Original  : Patient is a 45-year-old man with history of diabetes and hypertension!!!
Processed : ['patient', '45', 'year', 'old', 'man', 'history', 'diabetes', 'hypertension']
As string : patient 45 year old man history diabetes hypertension


In [ ]:
import ir_datasets
import json
from tqdm import tqdm

# ---- Process Dataset 1: Clinical Trials ----
print('Processing Dataset 1: clinicaltrials/2021/trec-ct-2021')
dataset1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')

processed_docs1 = {}
for doc in tqdm(dataset1.docs_iter(), desc='Docs CT2021'):
    # combine title + condition + summary fields
    raw_text = ' '.join(filter(None, [
        doc.title or '',
        str(doc.condition) if doc.condition else '',
        str(doc.intervention) if doc.intervention else '',
        str(doc.summary) if doc.summary else ''
    ]))
    processed_docs1[doc.doc_id] = preprocess_to_string(raw_text)

print(f'Processed {len(processed_docs1):,} documents')

# save
with open(f'{SAVE_DIR}/ct2021_docs_processed.json', 'w') as f:
    json.dump(processed_docs1, f)
print('Saved ct2021_docs_processed.json')

[INFO] [starting] building docstore


Processing Dataset 1: clinicaltrials/2021/trec-ct-2021


[INFO] If you have a local copy of http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/e12eb9a0d21452503b0ef8874c69f490
[INFO] [starting] http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip
docs_iter:   0%|                                    | 0/375580 [00:00<?, ?doc/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.0%| 0.00/383M [00:00<?, ?B/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.0%| 106k/383M [00:00<06:44, 947kB/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 0.2%| 672k/383M [00:00<02:11, 2.91MB/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 1.1%| 4.17M/383M [00:00<00:32, 11.8MB/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part1.zip: 2.2%| 8.42M/383M [00:00<00:20, 17.9MB/s]
http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.

In [ ]:
# Process Dataset 1 queries
processed_queries1 = {}
for query in dataset1.queries_iter():
    processed_queries1[query.query_id] = preprocess_to_string(query.text)

print(f'Processed {len(processed_queries1)} queries')

# save qrels as-is (no preprocessing needed)
qrels1 = {}
for qrel in dataset1.qrels_iter():
    if qrel.query_id not in qrels1:
        qrels1[qrel.query_id] = {}
    qrels1[qrel.query_id][qrel.doc_id] = qrel.relevance

with open(f'{SAVE_DIR}/ct2021_queries_processed.json', 'w') as f:
    json.dump(processed_queries1, f)
with open(f'{SAVE_DIR}/ct2021_qrels.json', 'w') as f:
    json.dump(qrels1, f)

print(f'Saved queries ({len(processed_queries1)}) and qrels ({len(qrels1)})')

In [ ]:
# ---- Process Dataset 2: MSMARCO ----
print('Processing Dataset 2: msmarco-passage/trec-dl-2019')
dataset2 = ir_datasets.load('msmarco-passage/trec-dl-2019')

processed_docs2 = {}
for doc in tqdm(dataset2.docs_iter(), desc='Docs MSMARCO'):
    processed_docs2[doc.doc_id] = preprocess_to_string(doc.text)

print(f'Processed {len(processed_docs2):,} documents')

with open(f'{SAVE_DIR}/msmarco_docs_processed.json', 'w') as f:
    json.dump(processed_docs2, f)
print('Saved msmarco_docs_processed.json')

In [ ]:
# Process Dataset 2 queries + qrels
processed_queries2 = {}
for query in dataset2.queries_iter():
    processed_queries2[query.query_id] = preprocess_to_string(query.text)

qrels2 = {}
for qrel in dataset2.qrels_iter():
    if qrel.query_id not in qrels2:
        qrels2[qrel.query_id] = {}
    qrels2[qrel.query_id][qrel.doc_id] = qrel.relevance

with open(f'{SAVE_DIR}/msmarco_queries_processed.json', 'w') as f:
    json.dump(processed_queries2, f)
with open(f'{SAVE_DIR}/msmarco_qrels.json', 'w') as f:
    json.dump(qrels2, f)

print(f'Queries: {len(processed_queries2)} | Qrels: {len(qrels2)}')

In [ ]:
# Verify output
print('=== Preprocessing Complete ===')
print()
print('Dataset 1 - Clinical Trials:')
sample_id = list(processed_docs1.keys())[0]
print(f'  Doc sample: {processed_docs1[sample_id][:80]}')
sample_qid = list(processed_queries1.keys())[0]
print(f'  Query sample: {processed_queries1[sample_qid]}')
print()
print('Dataset 2 - MSMARCO:')
sample_id2 = list(processed_docs2.keys())[0]
print(f'  Doc sample: {processed_docs2[sample_id2][:80]}')
sample_qid2 = list(processed_queries2.keys())[0]
print(f'  Query sample: {processed_queries2[sample_qid2]}')
print()
print('Next: 03_indexing.ipynb')